# AI Interview Training System - Qwen2.5-Coder-7B Fine-tuning

## Hướng dẫn hoàn chỉnh để train mô hình AI phỏng vấn trên Kaggle

Notebook này sẽ hướng dẫn bạn fine-tune mô hình Qwen/Qwen2.5-Coder-7B-Instruct để tạo ra một hệ thống AI phỏng vấn thông minh.

### Yêu cầu:
- Kaggle account với GPU enabled
- Internet connection để download model
- Khoảng 2-3 giờ để hoàn thành training

### Tính năng chính:
- LoRA fine-tuning cho hiệu quả memory
- Conversation format training
- Support 15+ technical positions
- Model evaluation và testing
- Export model cho production

## 1. Environment Setup và Dependencies

In [ ]:
# Install required packages
!pip install transformers==4.36.0
!pip install peft==0.7.0
!pip install datasets==2.14.0
!pip install accelerate==0.24.0
!pip install bitsandbytes==0.41.0
!pip install trl==0.7.0
!pip install torch==2.1.0
!pip install sentencepiece
!pip install protobuf

print("✅ All packages installed successfully!")

In [ ]:
# Import necessary libraries
import os
import torch
import json
import gc
from datetime import datetime
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Load và Prepare Training Data

In [ ]:
# Create sample training data (you can upload your own dataset to Kaggle)
sample_training_data = [
    {
        "cv_text": "Senior Python Developer with 6 years of experience in building scalable web applications. Proficient in Django, FastAPI, PostgreSQL, Redis, and Docker.",
        "job_description": "Senior Backend Developer. Requirements: Python, Django/FastAPI, PostgreSQL, Redis, Docker, AWS, microservices, API design, 5+ years experience.",
        "position": "Senior Backend Developer",
        "level": "senior",
        "questions": [
            {
                "question": "Explain Django ORM lazy loading and how to optimize N+1 query problems?",
                "category": "technical",
                "difficulty": "senior",
                "skill_focus": "Django ORM",
                "expected_keywords": ["select_related", "prefetch_related", "lazy loading", "N+1 problem"]
            }
        ]
    },
    {
        "cv_text": "Junior Frontend Developer with 1.5 years of experience building web applications with HTML, CSS, JavaScript, and React.",
        "job_description": "Junior Frontend Developer. Requirements: HTML, CSS, JavaScript, React basics, Git, responsive design, fresh graduate or 1-2 years experience.",
        "position": "Junior Frontend Developer", 
        "level": "junior",
        "questions": [
            {
                "question": "What is the difference between let, var, and const in JavaScript?",
                "category": "technical",
                "difficulty": "junior",
                "skill_focus": "JavaScript Fundamentals",
                "expected_keywords": ["var", "let", "const", "scope", "hoisting"]
            }
        ]
    }
]

# Save sample data
with open('training_data.json', 'w') as f:
    json.dump(sample_training_data, f, indent=2)

print(f"✅ Sample training data created with {len(sample_training_data)} examples")
print("📝 Để sử dụng dataset đầy đủ, upload file 'ai_interview_training_dataset.json' lên Kaggle Dataset")

In [ ]:
def create_conversation_format(example):
    """Convert training example to conversation format for Qwen"""
    cv_text = example['cv_text']
    job_description = example['job_description']
    position = example['position']
    level = example['level']
    
    # Create system prompt
    system_prompt = f"""
You are an expert technical interviewer. Your task is to generate relevant interview questions based on the candidate's CV and the job requirements.

Position: {position}
Level: {level}

Job Requirements:
{job_description}

Candidate CV:
{cv_text}

Generate appropriate technical interview questions that match the candidate's experience level and the job requirements.
""".strip()
    
    # Create conversation
    conversations = []
    
    for question_data in example['questions']:
        question = question_data['question']
        category = question_data['category']
        skill_focus = question_data['skill_focus']
        
        user_prompt = f"Generate a {category} interview question focusing on {skill_focus} for this {level} level candidate."
        
        conversation = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": question}
            ]
        }
        conversations.append(conversation)
    
    return conversations

# Load training data
with open('training_data.json', 'r') as f:
    training_data = json.load(f)

# Convert to conversation format
all_conversations = []
for example in training_data:
    conversations = create_conversation_format(example)
    all_conversations.extend(conversations)

print(f"✅ Created {len(all_conversations)} conversation examples")
print("📝 Example conversation:")
print(json.dumps(all_conversations[0], indent=2)[:500] + "...")

## 3. Model Setup với LoRA Configuration

In [ ]:
# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUTPUT_DIR = "./ai-interview-model"

# LoRA configuration for memory efficiency
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,  # rank
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

# Quantization config for memory efficiency 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("✅ Model configuration ready")
print(f"📝 Model: {MODEL_NAME}")
print(f"📝 LoRA rank: {lora_config.r}")
print(f"📝 Target modules: {lora_config.target_modules}")

In [ ]:
# Load tokenizer
print("🔄 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded. Vocab size: {len(tokenizer)}")
print(f"📝 Pad token: {tokenizer.pad_token}")
print(f"📝 EOS token: {tokenizer.eos_token}")

In [ ]:
# Load model with quantization
print("🔄 Loading model with quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Apply LoRA
print("🔄 Applying LoRA configuration...")
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"✅ Model loaded successfully!")
print(f"📝 Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"📝 Total parameters: {total_params:,}")

# Clear cache
torch.cuda.empty_cache()
gc.collect()

## 4. Data Preprocessing và Tokenization

In [ ]:
def format_conversation(conversation):
    """Format conversation for Qwen chat template"""
    formatted_text = ""
    
    for message in conversation["messages"]:
        role = message["role"]
        content = message["content"]
        
        if role == "system":
            formatted_text += f"<|im_start|>system\n{content}<|im_end|>\n"
        elif role == "user":
            formatted_text += f"<|im_start|>user\n{content}<|im_end|>\n"
        elif role == "assistant":
            formatted_text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
    
    return formatted_text

def tokenize_function(examples):
    """Tokenize formatted conversations"""
    texts = [format_conversation(conv) for conv in examples["conversation"]]
    
    # Tokenize with truncation and padding
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=1024,  # Adjust based on your data
        return_tensors="pt"
    )
    
    # For causal language modeling, labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].clone()
    
    return tokenized

# Create dataset
dataset_dict = {"conversation": all_conversations}
dataset = Dataset.from_dict(dataset_dict)

print(f"✅ Dataset created with {len(dataset)} examples")

# Tokenize dataset
print("🔄 Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Tokenizing"
)

print(f"✅ Dataset tokenized. Sample shape: {tokenized_dataset[0]['input_ids'].shape}")

# Split dataset (80/20 train/eval)
if len(tokenized_dataset) > 1:
    split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
else:
    train_dataset = tokenized_dataset
    eval_dataset = tokenized_dataset

print(f"📝 Training examples: {len(train_dataset)}")
print(f"📝 Evaluation examples: {len(eval_dataset)}")

## 5. Training Configuration và Fine-tuning

In [ ]:
# Training arguments optimized for Kaggle environment
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,  # Small batch size for memory efficiency
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # Effective batch size = 1 * 8 = 8
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,                       # Mixed precision training
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    dataloader_pin_memory=False,     # Disable for memory efficiency
    remove_unused_columns=False,
    report_to=None,                  # Disable wandb
)

print("✅ Training arguments configured")
print(f"📝 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📝 Total epochs: {training_args.num_train_epochs}")
print(f"📝 Learning rate: {training_args.learning_rate}")

In [ ]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal language modeling
    pad_to_multiple_of=8  # For efficiency
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("✅ Trainer initialized")
print(f"📝 Model device: {next(model.parameters()).device}")

# Print memory usage
if torch.cuda.is_available():
    print(f"📝 GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"📝 GPU Memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
# Start training
print("🚀 Starting training...")
print(f"⏰ Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Clear cache before training
torch.cuda.empty_cache()
gc.collect()

try:
    training_result = trainer.train()
    print("✅ Training completed successfully!")
    print(f"⏰ End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"📊 Final training loss: {training_result.training_loss:.4f}")
    
except Exception as e:
    print(f"❌ Training failed: {str(e)}")
    # Print memory info for debugging
    if torch.cuda.is_available():
        print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU Memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    raise

## 6. Model Evaluation và Testing

In [ ]:
# Evaluation
print("🔄 Evaluating model...")
eval_results = trainer.evaluate()

print("📊 Evaluation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

# Save evaluation results
with open(os.path.join(OUTPUT_DIR, "eval_results.json"), "w") as f:
    json.dump(eval_results, f, indent=2)

print("✅ Evaluation completed")

In [ ]:
# Test generation with the fine-tuned model
def test_interview_generation(cv_text, job_description, position, level, skill_focus="Python"):
    """Test interview question generation"""
    
    system_prompt = f"""
You are an expert technical interviewer. Your task is to generate relevant interview questions based on the candidate's CV and the job requirements.

Position: {position}
Level: {level}

Job Requirements:
{job_description}

Candidate CV:
{cv_text}

Generate appropriate technical interview questions that match the candidate's experience level and the job requirements.
""".strip()
    
    user_prompt = f"Generate a technical interview question focusing on {skill_focus} for this {level} level candidate."
    
    # Format conversation
    formatted_input = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"
    
    # Tokenize
    inputs = tokenizer(formatted_input, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract assistant response
    assistant_start = response.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    if assistant_start > len("<|im_start|>assistant\n") - 1:
        generated_question = response[assistant_start:].split("<|im_end|>")[0].strip()
    else:
        generated_question = "Unable to generate question"
    
    return generated_question

# Test examples
test_cases = [
    {
        "cv_text": "Senior Python Developer with 5 years experience in Django, FastAPI, and PostgreSQL",
        "job_description": "Senior Backend Developer. Requirements: Python, Django, PostgreSQL, API design",
        "position": "Senior Backend Developer", 
        "level": "senior",
        "skill_focus": "Django"
    },
    {
        "cv_text": "Junior Frontend Developer with 1 year experience in React and JavaScript",
        "job_description": "Junior Frontend Developer. Requirements: React, JavaScript, HTML, CSS",
        "position": "Junior Frontend Developer",
        "level": "junior", 
        "skill_focus": "React"
    }
]

print("🧪 Testing interview question generation:")
print("="*50)

for i, test_case in enumerate(test_cases, 1):
    print(f"\n📝 Test Case {i}:")
    print(f"Position: {test_case['position']}")
    print(f"Level: {test_case['level']}")
    print(f"Skill Focus: {test_case['skill_focus']}")
    
    generated_question = test_interview_generation(**test_case)
    print(f"\n🤖 Generated Question: {generated_question}")
    print("-"*30)

## 7. Save và Export Model

In [ ]:
# Save the fine-tuned model
print("💾 Saving fine-tuned model...")

# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Model saved to {OUTPUT_DIR}")

# Create model info file
model_info = {
    "base_model": MODEL_NAME,
    "training_date": datetime.now().isoformat(),
    "lora_config": {
        "r": lora_config.r,
        "alpha": lora_config.lora_alpha,
        "dropout": lora_config.lora_dropout,
        "target_modules": lora_config.target_modules
    },
    "training_params": {
        "epochs": training_args.num_train_epochs,
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps
    },
    "dataset_info": {
        "train_examples": len(train_dataset),
        "eval_examples": len(eval_dataset),
        "total_conversations": len(all_conversations)
    }
}

with open(os.path.join(OUTPUT_DIR, "model_info.json"), "w") as f:
    json.dump(model_info, f, indent=2)

print("📄 Model info saved")

# List saved files
print("\n📁 Saved files:")
for file in os.listdir(OUTPUT_DIR):
    file_path = os.path.join(OUTPUT_DIR, file)
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / 1024 / 1024
        print(f"  {file} ({size_mb:.1f} MB)")

In [ ]:
# Create deployment package
import zipfile

print("📦 Creating deployment package...")

# Create zip file with model
with zipfile.ZipFile("ai_interview_model_deployment.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, OUTPUT_DIR)
            zipf.write(file_path, arcname)

zip_size = os.path.getsize("ai_interview_model_deployment.zip") / 1024 / 1024
print(f"✅ Deployment package created: ai_interview_model_deployment.zip ({zip_size:.1f} MB)")

# Create inference script
inference_script = '''
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

def load_interview_model(model_path):
    """Load the fine-tuned interview model"""
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-Coder-7B-Instruct",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # Load LoRA adapter
    model = PeftModel.from_pretrained(base_model, model_path)
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    return model, tokenizer

def generate_interview_question(model, tokenizer, cv_text, job_description, position, level, skill_focus):
    """Generate interview question based on CV and job requirements"""
    system_prompt = f"""
You are an expert technical interviewer. Generate relevant interview questions based on the candidate's CV and job requirements.

Position: {position}
Level: {level}
Job Requirements: {job_description}
Candidate CV: {cv_text}
""".strip()
    
    user_prompt = f"Generate a technical interview question focusing on {skill_focus} for this {level} level candidate."
    
    formatted_input = f"<|im_start|>system\\n{system_prompt}<|im_end|>\\n<|im_start|>user\\n{user_prompt}<|im_end|>\\n<|im_start|>assistant\\n"
    
    inputs = tokenizer(formatted_input, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_start = response.find("<|im_start|>assistant\\n") + len("<|im_start|>assistant\\n")
    if assistant_start > len("<|im_start|>assistant\\n") - 1:
        generated_question = response[assistant_start:].split("<|im_end|>")[0].strip()
    else:
        generated_question = "Unable to generate question"
    
    return generated_question

# Example usage
if __name__ == "__main__":
    model, tokenizer = load_interview_model("./ai-interview-model")
    
    question = generate_interview_question(
        model, tokenizer,
        cv_text="Senior Python Developer with 5 years experience",
        job_description="Backend Developer role requiring Python and Django",
        position="Senior Backend Developer",
        level="senior",
        skill_focus="Django"
    )
    
    print(f"Generated question: {question}")
'''

with open("inference_example.py", "w") as f:
    f.write(inference_script)

print("✅ Inference script created: inference_example.py")

## 8. Cleanup và Memory Management

In [ ]:
# Cleanup to free memory
print("🧹 Cleaning up memory...")

del model
del trainer
torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    print(f"📝 GPU Memory after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

print("✅ Cleanup completed")

## 9. Summary và Next Steps

### 🎉 Training hoàn thành!

**Files được tạo:**
- `ai-interview-model/`: Thư mục chứa LoRA adapter và tokenizer
- `ai_interview_model_deployment.zip`: Package để deploy
- `inference_example.py`: Script để sử dụng model
- `eval_results.json`: Kết quả evaluation
- `model_info.json`: Thông tin chi tiết về model

### 📝 Hướng dẫn sử dụng:

1. **Download model**: Download file `ai_interview_model_deployment.zip`
2. **Setup production**: Giải nén và setup theo hướng dẫn trong API documentation
3. **Integration**: Tích hợp vào landing page thông qua FastAPI service

### 🚀 Next Steps:

1. **Tạo FastAPI service** để serve model
2. **Tích hợp frontend** với React components
3. **Setup Docker containers** cho deployment
4. **Testing và optimization** trong production environment

### 📊 Model Performance:
- Kiểm tra `eval_results.json` để xem performance metrics
- Test với các CV và job description khác nhau
- Fine-tune hyperparameters nếu cần thiết

**🎯 Mục tiêu đạt được:**
- ✅ Fine-tuned Qwen2.5-Coder-7B với LoRA
- ✅ Support 15+ technical positions 
- ✅ Conversation format training
- ✅ Memory-efficient training trên Kaggle
- ✅ Ready-to-deploy model package